In [10]:
!pip install langchain-ollama

In [11]:
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,ToolMessage

In [12]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def get_conversion_factor(base_currency :str,target_currency :str) -> float:
    """This function fetches the curency conversion factor 
    btn base currency and targted currency
    """
    
    url=f"https://api.fastforex.io/fetch-one?from={base_currency}&to={target_currency}&api_key={api_key3}"
    try:
        response=requests.get(url)
        response.raise_for_status()
        return response.json()
    
    except Exception as e:
        print(f'Error is {e}')
        
@tool
def convert(base_currency_value:float , conversion_rate:Annotated[float,InjectedToolArg]) -> float:
    """This function multiples the given money(currency amount) as per the exchange or conversion rate"""
    return base_currency_value*conversion_rate

In [13]:
llm=ChatOllama(model='qwen2.5:3b-instruct')

In [14]:
llm_with_tools=llm.bind_tools([get_conversion_factor,convert])

In [15]:
query="What is the conversion factor between usd and npr, and on that basis, how much is $10 USD in npr?"

In [16]:
ai_message=llm_with_tools.invoke(query)

In [17]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'usd', 'target_currency': 'npr'},
  'id': 'b78a5a21-41f0-45c9-935e-c590b50607eb',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10,
   'base_currency': 'usd',
   'target_currency': 'npr'},
  'id': '5a6df08d-2f31-4864-b249-fc936b0ca9ab',
  'type': 'tool_call'}]